# Phase 4: Modeling — Full Faithful Port

**Purpose:** Port the original AIM-AHEAD modeling pipeline to enable a sensitivity analysis comparing:
- **Model A**: CatBoost trained on all features
- **Model B**: CatBoost restricted to the top 10 features identified in the original SHAP analysis

**Inputs:** `../data/processed/03_cleaned.parquet`

**Outputs:**
- `../data/processed/04_model_a_full.pkl` — full-feature model
- `../data/processed/04_model_b_top10.pkl` — top-10 feature model
- `../data/processed/04_model_a_results.json` — full-feature metrics
- `../data/processed/04_model_b_results.json` — top-10 metrics
- `../data/processed/04_X_test.parquet`, `04_y_test.parquet` — held-out test set for SHAP in Phase 5

**Methodology:** Faithful port of the original notebook's modeling cell, with adaptations:
- Loads the cleaned dataset from Phase 3 instead of computing in-memory
- Adds explicit save points for downstream phases
- Runs the screening framework once (Model A), then refits CatBoost on the top-10 feature subset (Model B) using the same train/test split and seed

**Top 10 features (from manuscript Figure 4):**
1. Wheezing/Whistling in Chest (Past Year) — `RDQ070` (or equivalent)
2. Close Relative Had Asthma (family history) — `MCQ300B` (or equivalent)
3. General Health Condition — `HSD010` (or equivalent)
4. FEV1/FVC Ratio (engineered from spirometry)
5. Times Received Healthcare (Past Year) — `HUQ051`
6. Health Now vs. 1 Year Ago — `HSQ500` (or equivalent)
7. Family History × Lung Function (engineered interaction)
8. Forced Expiratory Time (seconds) — `SPXNFET`
9. Crawl/Walk/Run/Play Limitations — `PFQ020` or `PFQ030` (or equivalent)
10. Airway Obstruction Indicator (engineered binary)

**Note:** Several of the top 10 are *engineered features* (created in the original `ClinicalFeatureEngineer` transformer), not raw NHANES variables. The Model B subset will need to extract these post-transformation rather than pre-filtering raw columns.

In [3]:
"""
Load the Phase 3 cleaned dataset and prepare it for the modeling pipeline.

Removes:
- NHANES_CYCLE: provenance tag added in Phase 1 (string), not a feature
- Any other non-numeric columns the original modeling cell wasn't designed to handle
"""

from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("../data/processed")

split_df = pd.read_parquet(PROCESSED_DIR / "03_cleaned.parquet")

# Drop non-numeric provenance columns added during the rebuild
# that weren't present in the original pipeline
provenance_cols = ['NHANES_CYCLE']
existing_provenance = [c for c in provenance_cols if c in split_df.columns]
if existing_provenance:
    split_df = split_df.drop(columns=existing_provenance)
    print(f"Dropped provenance columns: {existing_provenance}")

# Sanity check: report any remaining non-numeric columns so we can spot issues early
non_numeric = split_df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f"WARNING: non-numeric columns still present: {non_numeric}")
else:
    print("All remaining columns are numeric.")

print()
print(f"Loaded shape: {split_df.shape[0]:,} rows × {split_df.shape[1]} columns")
print()
print(f"Outcome (MCQ010) distribution:")
print(split_df['MCQ010'].value_counts(dropna=False).to_string())
print()
print(f"WTMEC2YR present: {'WTMEC2YR' in split_df.columns}")
print(f"RIDAGEYR present: {'RIDAGEYR' in split_df.columns}")

Dropped provenance columns: ['NHANES_CYCLE']
All remaining columns are numeric.

Loaded shape: 6,784 rows × 98 columns

Outcome (MCQ010) distribution:
MCQ010
2.0    5524
1.0    1260

WTMEC2YR present: True
RIDAGEYR present: True


In [4]:
"""
COMPREHENSIVE PEDIATRIC ASTHMA SCREENING PIPELINE - NO DATA LEAKAGE
Corrects all methodological issues:
1. Split FIRST, before any feature engineering
2. All statistics computed from training data only
3. Leaky variables dropped BEFORE missingness features
4. Model selection on VALIDATION set only
5. Test set used ONCE for final reporting

Valid for ages 6-17 without age-restricted variables
"""

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, AdaBoostClassifier,
    HistGradientBoostingClassifier, VotingClassifier,
    StackingClassifier, BaggingClassifier
)
from sklearn.linear_model import (
    LogisticRegression, RidgeClassifier, SGDClassifier,
    ElasticNet, LassoCV, ElasticNetCV, PassiveAggressiveClassifier
)
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif, mutual_info_classif,
    SelectFromModel, RFE, RFECV, VarianceThreshold
)
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, brier_score_loss,
    average_precision_score, matthews_corrcoef, roc_curve,
    f1_score, cohen_kappa_score, balanced_accuracy_score
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.utils.class_weight import compute_sample_weight
from scipy.special import expit
from scipy import stats
import warnings
import time
from datetime import datetime

try:
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    ITERATIVE_IMPUTER_AVAILABLE = True
except:
    ITERATIVE_IMPUTER_AVAILABLE = False

warnings.filterwarnings('ignore')

# ===========================================================================
# CONDITIONAL IMPORTS
# ===========================================================================

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("Warning: XGBoost not available")

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("Warning: LightGBM not available")

try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("Warning: CatBoost not available")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    EBM_AVAILABLE = True
except ImportError:
    EBM_AVAILABLE = False

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    TABNET_AVAILABLE = True
except ImportError:
    TABNET_AVAILABLE = False

try:
    from boruta import BorutaPy
    BORUTA_AVAILABLE = True
except ImportError:
    BORUTA_AVAILABLE = False

try:
    from imblearn.ensemble import BalancedRandomForestClassifier, RUSBoostClassifier, EasyEnsembleClassifier
    from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours, NearMiss
    from imblearn.combine import SMOTETomek, SMOTEENN
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False
    print("Warning: imbalanced-learn not available")

# ===========================================================================
# CONFIGURATION
# ===========================================================================

# Asthma follow-up questions (post-diagnosis)
LEAKY_PROXIES = ["MCQ025", "MCQ035", "MCQ040", "MCQ050", "MCQ051"]

# Age-restricted variables (0-15 years only)
AGE_RESTRICTED_VARS = [
    'ECQ020', 'ECQ080', 'ECQ090', 'WHQ030E', 'MCQ080E', 
    'ECQ150', 'ECD010', 'ECD070A', 'ECD070B',
    'FSD670ZC', 'FSQ690', 'FSD680', 'FSD675'
]

# Identifiers to drop
IDENTIFIERS = ['SEQN']

# ===========================================================================
# NHANES DATA CLEANING TRANSFORMER
# ===========================================================================

class NHANESCleaner(BaseEstimator, TransformerMixin):
    """Clean NHANES sentinel codes - learns rules from training data"""
    
    def __init__(self):
        self.continuous_cols_ = None
        self.categorical_cols_ = None
        self.binary_cols_ = None
        
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Known continuous variables (from domain knowledge)
        continuous_whitelist = {
            'BMXBMI', 'BMXHT', 'BMXWT', 'BMXWAIST', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXHEAD',
            'BMXRECUM', 'BMXSAD1', 'BMXSAD2', 'BMXSAD3', 'BMXSAD4', 'BMXTHICR', 'BMXTRI',
            'BMXSUB', 'BMXCALF', 'SPXNFEV1', 'SPXNFVC', 'SPXNFEV3', 'SPXNFEV6', 'SPXNPEF', 
            'SPXNFET', 'SPXBFEV1', 'SPXBFVC', 'SPXBFEV3', 'SPXBFEV6', 'SPXBPEF',
            'LBXCOT', 'LBXTHC', 'URXUCR', 'URXNAL', 'URXUHG',
            'INDFMPIR', 'RIDAGEYR', 'INDHHIN2', 'DMDEDUC2', 'DMDEDUC3',
            'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXEOPCT', 'LBXBAPCT',
            'LBXIRN', 'LBXSGL', 'LBXGLU', 'LBDGLUSI'
        }
        
        self.continuous_cols_ = []
        self.categorical_cols_ = []
        self.binary_cols_ = []
        
        for col in X_df.columns:
            if col in continuous_whitelist:
                self.continuous_cols_.append(col)
            else:
                s = X_df[col]
                if pd.api.types.is_numeric_dtype(s):
                    vals = s.dropna()
                    if len(vals) > 0:
                        # Check if binary (1/2 coding common in NHANES)
                        unique_vals = set(vals.unique())
                        if unique_vals == {1, 2} or unique_vals == {1.0, 2.0}:
                            self.binary_cols_.append(col)
                        else:
                            near_int = np.all(np.abs(vals - np.round(vals)) < 1e-6)
                            unique_count = len(vals.unique())
                            if near_int and unique_count <= 20:
                                self.categorical_cols_.append(col)
                            else:
                                self.continuous_cols_.append(col)
        
        return self
    
    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        big_sentinels = {99, 999, 777, 7777, 9999, 77777, 99999}
        small_sentinels = {7, 9, 77}
        
        # Clean continuous columns (big sentinels only)
        for col in self.continuous_cols_:
            if col in X_df.columns:
                X_df[col] = X_df[col].replace(big_sentinels, np.nan)
        
        # Clean categorical columns (both big and small sentinels)
        for col in self.categorical_cols_:
            if col in X_df.columns:
                X_df[col] = X_df[col].replace(big_sentinels | small_sentinels, np.nan)
        
        # Convert binary variables (1=Yes/Male, 2=No/Female → 1=Yes/Female, 0=No/Male)
        for col in self.binary_cols_:
            if col in X_df.columns:
                X_df[col] = (X_df[col] == 1).astype(float)
                X_df.loc[X[col].isna(), col] = np.nan
        
        return X_df

# ===========================================================================
# COMPREHENSIVE FEATURE ENGINEERING TRANSFORMER
# ===========================================================================

class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    """Create clinical features - all statistics learned from training data"""
    
    def __init__(self):
        self.bmi_mean_ = None
        self.bmi_std_ = None
        self.race_categories_ = None
        self.feature_names_ = None
        
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Learn BMI statistics from training data only
        if 'BMXBMI' in X_df.columns:
            self.bmi_mean_ = X_df['BMXBMI'].mean()
            self.bmi_std_ = X_df['BMXBMI'].std()
            if self.bmi_std_ == 0 or pd.isna(self.bmi_std_):
                self.bmi_std_ = 1.0
        
        # Learn race categories from training data only
        if 'RIDRETH1' in X_df.columns:
            self.race_categories_ = sorted(X_df['RIDRETH1'].dropna().unique())
        
        return self
    
    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # ==================== SPIROMETRY FEATURES ====================
        if 'SPXNFEV1' in X_df.columns and 'SPXNFVC' in X_df.columns:
            mask = (X_df['SPXNFEV1'].notna() & X_df['SPXNFVC'].notna() & (X_df['SPXNFVC'] > 0))
            
            # Primary ratio
            X_df['fev1_fvc_ratio'] = np.nan
            X_df.loc[mask, 'fev1_fvc_ratio'] = X_df.loc[mask, 'SPXNFEV1'] / X_df.loc[mask, 'SPXNFVC']
            X_df['fev1_fvc_ratio_missing'] = (~mask).astype(int)
            
            # Clinical thresholds
            X_df['obstruction_indicator'] = (X_df['fev1_fvc_ratio'] < 0.8).astype(float)
            X_df['obstruction_mild'] = ((X_df['fev1_fvc_ratio'] >= 0.7) & (X_df['fev1_fvc_ratio'] < 0.8)).astype(float)
            X_df['obstruction_moderate'] = ((X_df['fev1_fvc_ratio'] >= 0.6) & (X_df['fev1_fvc_ratio'] < 0.7)).astype(float)
            X_df['obstruction_severe'] = (X_df['fev1_fvc_ratio'] < 0.6).astype(float)
            
            # FEV1 transformations
            X_df['fev1_log'] = np.log1p(X_df['SPXNFEV1'].fillna(0))
            X_df['fev1_sqrt'] = np.sqrt(X_df['SPXNFEV1'].fillna(0))
            X_df['fev1_squared'] = X_df['SPXNFEV1'].fillna(0) ** 2
            
            # FVC transformations
            X_df['fvc_log'] = np.log1p(X_df['SPXNFVC'].fillna(0))
            X_df['fvc_sqrt'] = np.sqrt(X_df['SPXNFVC'].fillna(0))
            
            # Additional spirometry ratios
            if 'SPXNFEV6' in X_df.columns:
                mask6 = (X_df['SPXNFEV6'] > 0) & (X_df['SPXNFEV1'].notna())
                X_df['fev1_fev6_ratio'] = np.nan
                X_df.loc[mask6, 'fev1_fev6_ratio'] = X_df.loc[mask6, 'SPXNFEV1'] / X_df.loc[mask6, 'SPXNFEV6']
            
            if 'SPXNFEV3' in X_df.columns:
                mask3 = (X_df['SPXNFEV3'] > 0) & (X_df['SPXNFEV1'].notna())
                X_df['fev1_fev3_ratio'] = np.nan
                X_df.loc[mask3, 'fev1_fev3_ratio'] = X_df.loc[mask3, 'SPXNFEV1'] / X_df.loc[mask3, 'SPXNFEV3']
            
            if 'SPXNPEF' in X_df.columns:
                X_df['pef_log'] = np.log1p(X_df['SPXNPEF'].fillna(0))
                pef_mask = (X_df['SPXNPEF'] > 0) & (X_df['SPXNFEV1'] > 0)
                X_df['fev1_pef_ratio'] = np.nan
                X_df.loc[pef_mask, 'fev1_pef_ratio'] = X_df.loc[pef_mask, 'SPXNFEV1'] / X_df.loc[pef_mask, 'SPXNPEF']
            
            # Age-adjusted spirometry
            if 'RIDAGEYR' in X_df.columns:
                age_mask = X_df['RIDAGEYR'] > 0
                X_df['fev1_age_adjusted'] = np.nan
                X_df.loc[age_mask, 'fev1_age_adjusted'] = X_df.loc[age_mask, 'SPXNFEV1'] / X_df.loc[age_mask, 'RIDAGEYR']
                X_df['fvc_age_adjusted'] = np.nan
                X_df.loc[age_mask, 'fvc_age_adjusted'] = X_df.loc[age_mask, 'SPXNFVC'] / X_df.loc[age_mask, 'RIDAGEYR']
                X_df['fev1_age_interaction'] = X_df['SPXNFEV1'].fillna(0) * X_df['RIDAGEYR'].fillna(0)
            
            # Height-adjusted spirometry
            if 'BMXHT' in X_df.columns:
                height_mask = X_df['BMXHT'] > 0
                X_df['fev1_height_adjusted'] = np.nan
                X_df.loc[height_mask, 'fev1_height_adjusted'] = X_df.loc[height_mask, 'SPXNFEV1'] / (X_df.loc[height_mask, 'BMXHT'] / 100)
                X_df['fvc_height_adjusted'] = np.nan
                X_df.loc[height_mask, 'fvc_height_adjusted'] = X_df.loc[height_mask, 'SPXNFVC'] / (X_df.loc[height_mask, 'BMXHT'] / 100)
            
            # Bronchodilator response
            if 'SPXBFEV1' in X_df.columns:
                bd_mask = (X_df['SPXBFEV1'].notna() & X_df['SPXNFEV1'].notna() & (X_df['SPXNFEV1'] > 0))
                X_df['bronchodilator_response'] = np.nan
                X_df.loc[bd_mask, 'bronchodilator_response'] = ((X_df.loc[bd_mask, 'SPXBFEV1'] - X_df.loc[bd_mask, 'SPXNFEV1']) / 
                                                                  X_df.loc[bd_mask, 'SPXNFEV1']) * 100
                X_df['positive_bd_response'] = (X_df['bronchodilator_response'] >= 12).astype(float)
        
        # ==================== COTININE / SMOKE EXPOSURE ====================
        if 'LBXCOT' in X_df.columns:
            X_df['cotinine_log'] = np.log1p(X_df['LBXCOT'].fillna(0))
            X_df['cotinine_sqrt'] = np.sqrt(X_df['LBXCOT'].fillna(0))
            X_df['cotinine_squared'] = X_df['LBXCOT'].fillna(0) ** 2
            X_df['cotinine_missing'] = X_df['LBXCOT'].isna().astype(int)
            
            # Clinical thresholds
            X_df['no_smoke_exposure'] = (X_df['LBXCOT'] <= 0.05).astype(float)
            X_df['smoke_exposure_light'] = ((X_df['LBXCOT'] > 0.05) & (X_df['LBXCOT'] <= 1.0)).astype(float)
            X_df['smoke_exposure_moderate'] = ((X_df['LBXCOT'] > 1.0) & (X_df['LBXCOT'] <= 10.0)).astype(float)
            X_df['smoke_exposure_heavy'] = (X_df['LBXCOT'] > 10.0).astype(float)
            X_df['smoke_exposure_binary'] = (X_df['LBXCOT'] > 1.0).astype(float)
            X_df['likely_active_smoking'] = (X_df['LBXCOT'] > 50).astype(float)
            X_df['likely_passive_smoking'] = ((X_df['LBXCOT'] > 1.0) & (X_df['LBXCOT'] <= 50)).astype(float)
        
        # ==================== ANTHROPOMETRIC FEATURES ====================
        if 'BMXBMI' in X_df.columns:
            X_df['bmi_log'] = np.log1p(X_df['BMXBMI'].fillna(0))
            X_df['bmi_sqrt'] = np.sqrt(X_df['BMXBMI'].fillna(0))
            X_df['bmi_squared'] = X_df['BMXBMI'].fillna(0) ** 2
            X_df['bmi_missing'] = X_df['BMXBMI'].isna().astype(int)
            
            # BMI categories
            X_df['underweight'] = (X_df['BMXBMI'] < 18.5).astype(float)
            X_df['normal_weight'] = ((X_df['BMXBMI'] >= 18.5) & (X_df['BMXBMI'] < 25)).astype(float)
            X_df['overweight'] = ((X_df['BMXBMI'] >= 25) & (X_df['BMXBMI'] < 30)).astype(float)
            X_df['obese'] = (X_df['BMXBMI'] >= 30).astype(float)
            X_df['severely_obese'] = (X_df['BMXBMI'] >= 35).astype(float)
            
            # BMI Z-score using TRAINING statistics
            if self.bmi_mean_ is not None and self.bmi_std_ is not None:
                X_df['bmi_zscore'] = (X_df['BMXBMI'] - self.bmi_mean_) / self.bmi_std_
            
            # Age-adjusted BMI
            if 'RIDAGEYR' in X_df.columns:
                age_bmi_mask = (X_df['BMXBMI'].notna() & X_df['RIDAGEYR'].notna() & (X_df['RIDAGEYR'] > 0))
                X_df['bmi_age_adjusted'] = np.nan
                X_df.loc[age_bmi_mask, 'bmi_age_adjusted'] = X_df.loc[age_bmi_mask, 'BMXBMI'] / X_df.loc[age_bmi_mask, 'RIDAGEYR']
                X_df['bmi_age_interaction'] = X_df['BMXBMI'].fillna(0) * X_df['RIDAGEYR'].fillna(0)
        
        # Height features
        if 'BMXHT' in X_df.columns:
            X_df['height_log'] = np.log1p(X_df['BMXHT'].fillna(0))
            X_df['height_squared'] = X_df['BMXHT'].fillna(0) ** 2
            
            if 'RIDAGEYR' in X_df.columns:
                age_ht_mask = (X_df['BMXHT'].notna() & X_df['RIDAGEYR'].notna() & (X_df['RIDAGEYR'] > 0))
                X_df['height_age_ratio'] = np.nan
                X_df.loc[age_ht_mask, 'height_age_ratio'] = X_df.loc[age_ht_mask, 'BMXHT'] / X_df.loc[age_ht_mask, 'RIDAGEYR']
        
        # Weight features
        if 'BMXWT' in X_df.columns:
            X_df['weight_log'] = np.log1p(X_df['BMXWT'].fillna(0))
            X_df['weight_sqrt'] = np.sqrt(X_df['BMXWT'].fillna(0))
            
            if 'RIDAGEYR' in X_df.columns:
                age_wt_mask = (X_df['BMXWT'].notna() & X_df['RIDAGEYR'].notna() & (X_df['RIDAGEYR'] > 0))
                X_df['weight_age_ratio'] = np.nan
                X_df.loc[age_wt_mask, 'weight_age_ratio'] = X_df.loc[age_wt_mask, 'BMXWT'] / X_df.loc[age_wt_mask, 'RIDAGEYR']
        
        # Waist features
        if 'BMXWAIST' in X_df.columns:
            X_df['waist_log'] = np.log1p(X_df['BMXWAIST'].fillna(0))
            
            if 'BMXHT' in X_df.columns:
                waist_ht_mask = (X_df['BMXWAIST'] > 0) & (X_df['BMXHT'] > 0)
                X_df['waist_height_ratio'] = np.nan
                X_df.loc[waist_ht_mask, 'waist_height_ratio'] = X_df.loc[waist_ht_mask, 'BMXWAIST'] / X_df.loc[waist_ht_mask, 'BMXHT']
                X_df['waist_height_risk'] = (X_df['waist_height_ratio'] > 0.5).astype(float)
        
        # ==================== FAMILY HISTORY INTERACTIONS ====================
        if 'MCQ300B' in X_df.columns:
            X_df['MCQ300B_missing'] = X_df['MCQ300B'].isna().astype(int)
            
            if 'fev1_fvc_ratio' in X_df.columns:
                X_df['family_spirometry_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['fev1_fvc_ratio'].fillna(0)
                X_df['family_obstruction_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['obstruction_indicator'].fillna(0)
            
            if 'smoke_exposure_heavy' in X_df.columns:
                X_df['family_smoke_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['smoke_exposure_heavy'].fillna(0)
            if 'cotinine_log' in X_df.columns:
                X_df['family_cotinine_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['cotinine_log'].fillna(0)
            
            if 'obese' in X_df.columns:
                X_df['family_obesity_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['obese'].fillna(0)
            if 'BMXBMI' in X_df.columns:
                X_df['family_bmi_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['BMXBMI'].fillna(0)
            
            if 'RIDAGEYR' in X_df.columns:
                X_df['family_age_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['RIDAGEYR'].fillna(0)
        
        # ==================== AGE FEATURES ====================
        if 'RIDAGEYR' in X_df.columns:
            X_df['age_squared'] = X_df['RIDAGEYR'] ** 2
            X_df['age_cubed'] = X_df['RIDAGEYR'] ** 3
            X_df['age_log'] = np.log1p(X_df['RIDAGEYR'])
            X_df['age_sqrt'] = np.sqrt(X_df['RIDAGEYR'])
            
            # Age groups
            X_df['age_group_6_8'] = ((X_df['RIDAGEYR'] >= 6) & (X_df['RIDAGEYR'] < 9)).astype(float)
            X_df['age_group_9_12'] = ((X_df['RIDAGEYR'] >= 9) & (X_df['RIDAGEYR'] < 13)).astype(float)
            X_df['age_group_13_15'] = ((X_df['RIDAGEYR'] >= 13) & (X_df['RIDAGEYR'] < 16)).astype(float)
            X_df['age_group_16_17'] = ((X_df['RIDAGEYR'] >= 16) & (X_df['RIDAGEYR'] < 18)).astype(float)
            
            # Developmental stages
            X_df['pre_pubertal'] = (X_df['RIDAGEYR'] < 11).astype(float)
            X_df['likely_pubertal'] = ((X_df['RIDAGEYR'] >= 11) & (X_df['RIDAGEYR'] <= 15)).astype(float)
            X_df['post_pubertal'] = (X_df['RIDAGEYR'] > 15).astype(float)
            
            X_df['early_adolescent'] = ((X_df['RIDAGEYR'] >= 10) & (X_df['RIDAGEYR'] < 13)).astype(float)
            X_df['mid_adolescent'] = ((X_df['RIDAGEYR'] >= 13) & (X_df['RIDAGEYR'] < 16)).astype(float)
            X_df['late_adolescent'] = (X_df['RIDAGEYR'] >= 16).astype(float)
        
        # ==================== GENDER INTERACTIONS ====================
        if 'RIAGENDR' in X_df.columns:
            if 'RIDAGEYR' in X_df.columns:
                X_df['gender_age_interaction'] = X_df['RIAGENDR'].fillna(0) * X_df['RIDAGEYR'].fillna(0)
                X_df['female_adolescent'] = (X_df['RIAGENDR'].fillna(0) * X_df['likely_pubertal'].fillna(0))
            
            if 'obese' in X_df.columns:
                X_df['gender_obesity_interaction'] = X_df['RIAGENDR'].fillna(0) * X_df['obese'].fillna(0)
            
            if 'smoke_exposure_heavy' in X_df.columns:
                X_df['gender_smoke_interaction'] = X_df['RIAGENDR'].fillna(0) * X_df['smoke_exposure_heavy'].fillna(0)
            
            if 'fev1_fvc_ratio' in X_df.columns:
                X_df['gender_spirometry_interaction'] = X_df['RIAGENDR'].fillna(0) * X_df['fev1_fvc_ratio'].fillna(0)
        
        # ==================== RACE/ETHNICITY FEATURES ====================
        if 'RIDRETH1' in X_df.columns and self.race_categories_ is not None:
            # Use only categories seen during training
            for race_val in self.race_categories_:
                X_df[f'race_{int(race_val)}'] = (X_df['RIDRETH1'] == race_val).astype(float)
            
            if 'smoke_exposure_heavy' in X_df.columns:
                X_df['race_smoke_interaction'] = X_df['RIDRETH1'].fillna(0) * X_df['smoke_exposure_heavy'].fillna(0)
        
        # ==================== SOCIOECONOMIC FEATURES ====================
        if 'INDFMPIR' in X_df.columns:
            X_df['poverty_income_ratio_log'] = np.log1p(X_df['INDFMPIR'].fillna(0))
            X_df['poverty_income_ratio_sqrt'] = np.sqrt(X_df['INDFMPIR'].fillna(0))
            X_df['poverty_income_ratio_missing'] = X_df['INDFMPIR'].isna().astype(int)
            
            # Poverty thresholds
            X_df['extreme_poverty'] = (X_df['INDFMPIR'] < 0.5).astype(float)
            X_df['poverty'] = ((X_df['INDFMPIR'] >= 0.5) & (X_df['INDFMPIR'] < 1.0)).astype(float)
            X_df['low_income'] = ((X_df['INDFMPIR'] >= 1.0) & (X_df['INDFMPIR'] < 2.0)).astype(float)
            X_df['middle_income'] = ((X_df['INDFMPIR'] >= 2.0) & (X_df['INDFMPIR'] < 4.0)).astype(float)
            X_df['high_income'] = (X_df['INDFMPIR'] >= 4.0).astype(float)
            
            if 'smoke_exposure_heavy' in X_df.columns:
                X_df['poverty_smoke_interaction'] = X_df['low_income'].fillna(0) * X_df['smoke_exposure_heavy'].fillna(0)
        
        # ==================== RESPIRATORY SYMPTOMS ====================
        if 'RDQ070' in X_df.columns:
            X_df['RDQ070_missing'] = X_df['RDQ070'].isna().astype(int)
            
            if 'fev1_fvc_ratio' in X_df.columns:
                X_df['wheeze_spirometry_interaction'] = X_df['RDQ070'].fillna(0) * X_df['fev1_fvc_ratio'].fillna(0)
            
            if 'smoke_exposure_heavy' in X_df.columns:
                X_df['wheeze_smoke_interaction'] = X_df['RDQ070'].fillna(0) * X_df['smoke_exposure_heavy'].fillna(0)
        
        # Other medical conditions
        for mcq_var in ['MCQ092', 'MCQ053', 'MCQ140']:
            if mcq_var in X_df.columns:
                X_df[f'{mcq_var}_missing'] = X_df[mcq_var].isna().astype(int)
        
        # Sleep features
        for sleep_var in ['SPQ010', 'SPQ040', 'SPQ050']:
            if sleep_var in X_df.columns:
                X_df[f'{sleep_var}_missing'] = X_df[sleep_var].isna().astype(int)
        
        # ==================== LABORATORY VALUES ====================
        if 'LBXWBCSI' in X_df.columns:
            X_df['wbc_log'] = np.log1p(X_df['LBXWBCSI'].fillna(0))
            X_df['wbc_elevated'] = (X_df['LBXWBCSI'] > 11.0).astype(float)
        
        if 'LBXEOPCT' in X_df.columns:
            X_df['eosinophil_elevated'] = (X_df['LBXEOPCT'] > 4.0).astype(float)
            X_df['eosinophil_log'] = np.log1p(X_df['LBXEOPCT'].fillna(0))
        
        # ==================== MISSING VALUE PATTERNS ====================
        # Only count missingness for NON-LEAKY variables
        important_vars = ['RDQ070', 'MCQ092', 'MCQ140', 'MCQ053', 'SPQ010', 'SPQ040',
                         'SPXNFEV1', 'SPXNFVC', 'LBXCOT', 'BMXBMI']
        
        for var in important_vars:
            if var in X_df.columns and f'{var}_missing' not in X_df.columns:
                X_df[f'{var}_missing'] = X_df[var].isna().astype(int)
        
        X_df['total_missing'] = X_df.isna().sum(axis=1)
        X_df['missing_proportion'] = X_df['total_missing'] / len(X_df.columns)
        X_df['has_spirometry'] = ((X_df['SPXNFEV1'].notna()) | (X_df['SPXNFVC'].notna())).astype(int)
        X_df['has_cotinine'] = X_df['LBXCOT'].notna().astype(int)
        X_df['has_bmi'] = X_df['BMXBMI'].notna().astype(int)
        X_df['complete_core_data'] = ((X_df['has_spirometry'] == 1) & 
                                       (X_df['has_cotinine'] == 1) & 
                                       (X_df['has_bmi'] == 1)).astype(int)
        
        # ==================== POLYNOMIAL INTERACTIONS ====================
        if 'RIDAGEYR' in X_df.columns and 'BMXBMI' in X_df.columns:
            X_df['age_bmi_poly'] = X_df['RIDAGEYR'].fillna(0) * X_df['BMXBMI'].fillna(0) ** 2
        
        if 'fev1_fvc_ratio' in X_df.columns and 'RIDAGEYR' in X_df.columns:
            X_df['spirometry_age_poly'] = X_df['fev1_fvc_ratio'].fillna(0) * X_df['RIDAGEYR'].fillna(0) ** 2
        
        self.feature_names_ = X_df.columns.tolist()
        
        return X_df
    
    def get_feature_names_out(self, input_features=None):
        return self.feature_names_ if self.feature_names_ is not None else []

# ===========================================================================
# FEATURE SELECTION
# ===========================================================================

class EnhancedFeatureSelector(BaseEstimator, TransformerMixin):
    """Feature selection - fit on training data only"""
    
    def __init__(self, method='rf_importance', n_features=20):
        self.method = method
        self.n_features = n_features
        self.selected_indices_ = None
        self.feature_scores_ = None
        
    def fit(self, X, y, sample_weight=None):
        n_features_actual = min(self.n_features, X.shape[1])
        
        if self.method == 'rf_importance':
            rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
            rf.fit(X, y, sample_weight=sample_weight)
            importances = rf.feature_importances_
            self.selected_indices_ = np.argsort(importances)[-n_features_actual:]
            self.feature_scores_ = importances
            
        elif self.method == 'lasso':
            try:
                lr = LogisticRegression(penalty='l1', solver='saga', C=0.5, max_iter=10000, random_state=42)
                lr.fit(X, y, sample_weight=sample_weight)
                coef_abs = np.abs(lr.coef_).ravel()
                self.selected_indices_ = np.argsort(coef_abs)[-n_features_actual:]
                self.feature_scores_ = coef_abs
            except:
                return self._rf_fallback(X, y, sample_weight, n_features_actual)
            
        elif self.method == 'elastic_net':
            try:
                lr = LogisticRegression(penalty='elasticnet', solver='saga', C=0.5, l1_ratio=0.5, 
                                       max_iter=10000, random_state=42)
                lr.fit(X, y, sample_weight=sample_weight)
                coef_abs = np.abs(lr.coef_).ravel()
                self.selected_indices_ = np.argsort(coef_abs)[-n_features_actual:]
                self.feature_scores_ = coef_abs
            except:
                return self._rf_fallback(X, y, sample_weight, n_features_actual)
            
        elif self.method == 'mutual_info':
            selector = SelectKBest(mutual_info_classif, k=n_features_actual)
            selector.fit(X, y)
            self.selected_indices_ = selector.get_support(indices=True)
            self.feature_scores_ = selector.scores_
            
        elif self.method == 'f_classif':
            selector = SelectKBest(f_classif, k=n_features_actual)
            selector.fit(X, y)
            self.selected_indices_ = selector.get_support(indices=True)
            self.feature_scores_ = selector.scores_
            
        elif self.method == 'combined_filter':
            scores = np.zeros(X.shape[1])
            mi_scores = mutual_info_classif(X, y, random_state=42)
            if mi_scores.max() > 0:
                scores += mi_scores / mi_scores.max()
            f_scores, _ = f_classif(X, y)
            f_scores = np.nan_to_num(f_scores, 0)
            if f_scores.max() > 0:
                scores += f_scores / f_scores.max()
            variances = np.var(X, axis=0)
            if variances.max() > 0:
                scores += variances / variances.max()
            self.selected_indices_ = np.argsort(scores)[-n_features_actual:]
            self.feature_scores_ = scores
            
        elif self.method == 'xgb_importance' and XGB_AVAILABLE:
            xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                         subsample=0.8, colsample_bytree=0.8, random_state=42,
                                         eval_metric='logloss', verbosity=0)
            xgb_model.fit(X, y, sample_weight=sample_weight)
            importances = xgb_model.feature_importances_
            self.selected_indices_ = np.argsort(importances)[-n_features_actual:]
            self.feature_scores_ = importances
            
        else:
            return self._rf_fallback(X, y, sample_weight, n_features_actual)
        
        return self
    
    def _rf_fallback(self, X, y, sample_weight, n_features):
        rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
        rf.fit(X, y, sample_weight=sample_weight)
        importances = rf.feature_importances_
        self.selected_indices_ = np.argsort(importances)[-n_features:]
        self.feature_scores_ = importances
        return self
    
    def transform(self, X):
        if self.selected_indices_ is not None:
            return X[:, self.selected_indices_]
        return X

# ===========================================================================
# IMBALANCE HANDLING
# ===========================================================================

def handle_enhanced_imbalance(X_train, y_train, sw_train, method='none'):
    if IMBLEARN_AVAILABLE and method in ['smote', 'smote_enn', 'borderline_smote', 'adasyn']:
        try:
            if method == 'smote':
                resampler = SMOTE(random_state=42)
            elif method == 'smote_enn':
                resampler = SMOTEENN(random_state=42)
            elif method == 'borderline_smote':
                resampler = BorderlineSMOTE(random_state=42, kind='borderline-1')
            elif method == 'adasyn':
                resampler = ADASYN(random_state=42)
            
            X_resampled, y_resampled = resampler.fit_resample(X_train, y_train)
            return X_resampled, y_resampled, np.ones(len(y_resampled))
        except:
            pass
    
    if method == 'balanced':
        class_weights = compute_sample_weight('balanced', y_train)
        combined_weights = sw_train * class_weights if sw_train is not None else class_weights
        return X_train, y_train, combined_weights
    
    elif method.startswith('cost_'):
        try:
            cost_ratio = float(method.split('_')[1].replace('x', ''))
        except:
            cost_ratio = 5.0
        weights = np.ones(len(y_train))
        weights[y_train == 1] *= cost_ratio
        combined_weights = sw_train * weights if sw_train is not None else weights
        return X_train, y_train, combined_weights
    
    else:
        return X_train, y_train, sw_train

# ===========================================================================
# ALGORITHMS
# ===========================================================================

def create_comprehensive_algorithms():
    algorithms = {}
    
    algorithms['LogisticRegression'] = LogisticRegression(max_iter=5000, random_state=42)
    algorithms['LogisticRegressionL1'] = LogisticRegression(penalty='l1', solver='saga', max_iter=5000, random_state=42)
    algorithms['RandomForest'] = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
    algorithms['ExtraTrees'] = ExtraTreesClassifier(n_estimators=100, max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
    algorithms['GradientBoosting'] = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8, random_state=42)
    algorithms['AdaBoost'] = AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)
    algorithms['HistGradientBoosting'] = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42)
    algorithms['DecisionTree'] = DecisionTreeClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
    algorithms['RidgeClassifier'] = RidgeClassifier(alpha=1.0, random_state=42)
    algorithms['SGDClassifier'] = SGDClassifier(loss='log_loss', penalty='elasticnet', l1_ratio=0.5, max_iter=5000, random_state=42)
    algorithms['GaussianNB'] = GaussianNB()
    algorithms['LDA'] = LinearDiscriminantAnalysis()
    algorithms['KNN'] = KNeighborsClassifier(n_neighbors=10, weights='distance')
    algorithms['MLP'] = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, early_stopping=True, random_state=42)
    
    if XGB_AVAILABLE:
        algorithms['XGBoost'] = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.8,
                                                  colsample_bytree=0.8, random_state=42, eval_metric='logloss', verbosity=0)
    
    if LGBM_AVAILABLE:
        algorithms['LightGBM'] = lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, 
                                                    random_state=42, verbosity=-1, force_col_wise=True)
    
    if CATBOOST_AVAILABLE:
        algorithms['CatBoost'] = cb.CatBoostClassifier(iterations=100, depth=6, verbose=False, random_state=42)
    
    if IMBLEARN_AVAILABLE:
        algorithms['BalancedRF'] = BalancedRandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    
    return algorithms

# ===========================================================================
# METRICS
# ===========================================================================

def calculate_comprehensive_metrics(y_true, y_pred_proba, sample_weight=None, threshold=0.5):
    metrics = {}
    y_true = np.array(y_true)
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    if sample_weight is not None:
        tp = np.sum(sample_weight[(y_true == 1) & (y_pred == 1)])
        fn = np.sum(sample_weight[(y_true == 1) & (y_pred == 0)])
        tn = np.sum(sample_weight[(y_true == 0) & (y_pred == 0)])
        fp = np.sum(sample_weight[(y_true == 0) & (y_pred == 1)])
    else:
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            return {'auc': np.nan}
    
    metrics['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0
    metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    metrics['ppv'] = tp / (tp + fp) if (tp + fp) > 0 else 0
    metrics['npv'] = tn / (tn + fn) if (tn + fn) > 0 else 0
    metrics['accuracy'] = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    metrics['nns'] = 1 / metrics['ppv'] if metrics['ppv'] > 0 else np.inf
    metrics['f1'] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    
    try:
        metrics['auc'] = roc_auc_score(y_true, y_pred_proba, sample_weight=sample_weight)
    except:
        metrics['auc'] = np.nan
    
    try:
        metrics['brier'] = brier_score_loss(y_true, y_pred_proba, sample_weight=sample_weight)
    except:
        metrics['brier'] = np.nan
    
    try:
        metrics['mcc'] = matthews_corrcoef(y_true, y_pred)
    except:
        metrics['mcc'] = np.nan
    
    n = tp + tn + fp + fn
    metrics['net_benefit'] = (tp/n) - (fp/n) * (threshold/(1-threshold)) if threshold < 1 and n > 0 else 0
    metrics['threshold'] = threshold
    
    return metrics

# ===========================================================================
# MAIN PIPELINE - NO LEAKAGE
# ===========================================================================

def evaluate_comprehensive_pipeline_no_leakage(X, y, sample_weight=None, min_sensitivity=0.80,
                                               use_calibration=True, quick_mode=False):
    
    print("="*80)
    print("COMPREHENSIVE PEDIATRIC ASTHMA PIPELINE - NO DATA LEAKAGE")
    print("="*80)
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Minimum sensitivity threshold: {min_sensitivity:.1%}")
    print(f"Mode: {'Quick' if quick_mode else 'Full'}")
    print("-"*80)
    
    # ==================== STEP 1: REMOVE LEAKY/RESTRICTED/ID COLUMNS ====================
    print("\n1. Removing leaky, age-restricted, and identifier columns...")
    
    X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
    
    # Remove columns
    cols_to_drop = []
    for col in X_df.columns:
        if col in LEAKY_PROXIES + AGE_RESTRICTED_VARS + IDENTIFIERS:
            cols_to_drop.append(col)
        # Also remove any feature that contains these variable names
        elif any(var.lower() in col.lower() for var in AGE_RESTRICTED_VARS):
            cols_to_drop.append(col)
    
    if cols_to_drop:
        print(f"   Dropping {len(cols_to_drop)} columns: {cols_to_drop[:10]}...")
        X_df = X_df.drop(columns=cols_to_drop)
    
    print(f"   Remaining columns: {X_df.shape[1]}")
    
    # Convert to arrays
    X_array = X_df.values
    y_array = y.values if hasattr(y, 'values') else np.array(y)
    sw_array = sample_weight.values if hasattr(sample_weight, 'values') else sample_weight
    
    # ==================== STEP 2: SPLIT FIRST ====================
    print("\n2. Splitting data BEFORE any feature engineering (60% train / 20% val / 20% test)...")
    
    if sw_array is not None:
        X_temp, X_test, y_temp, y_test, sw_temp, sw_test = train_test_split(
            X_array, y_array, sw_array, test_size=0.2, random_state=42, stratify=y_array
        )
        X_train, X_val, y_train, y_val, sw_train, sw_val = train_test_split(
            X_temp, y_temp, sw_temp, test_size=0.25, random_state=42, stratify=y_temp
        )
    else:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X_array, y_array, test_size=0.2, random_state=42, stratify=y_array
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
        )
        sw_train = sw_val = sw_test = None
    
    prevalence = np.average(y_train, weights=sw_train) if sw_train is not None else y_train.mean()
    print(f"   Training:   {len(X_train):,} samples")
    print(f"   Validation: {len(X_val):,} samples")
    print(f"   Test:       {len(X_test):,} samples")
    print(f"   Asthma prevalence: {prevalence:.1%}")
    
    # Restore column names for transformers
    X_train = pd.DataFrame(X_train, columns=X_df.columns)
    X_val = pd.DataFrame(X_val, columns=X_df.columns)
    X_test = pd.DataFrame(X_test, columns=X_df.columns)
    
    # ==================== STEP 3: FIT TRANSFORMERS ON TRAIN ONLY ====================
    print("\n3. Fitting preprocessing transformers on TRAINING data only...")
    
    # NHANES cleaning
    cleaner = NHANESCleaner()
    cleaner.fit(X_train)
    X_train_clean = cleaner.transform(X_train)
    X_val_clean = cleaner.transform(X_val)
    X_test_clean = cleaner.transform(X_test)
    
    # Feature engineering
    print("   Creating exhaustive clinical features...")
    feat_eng = ClinicalFeatureEngineer()
    feat_eng.fit(X_train_clean)
    X_train_feat = feat_eng.transform(X_train_clean)
    X_val_feat = feat_eng.transform(X_val_clean)
    X_test_feat = feat_eng.transform(X_test_clean)
    
    print(f"   Total features after engineering: {X_train_feat.shape[1]}")
    
    # Imputation
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_feat)
    X_val_imp = imputer.transform(X_val_feat)
    X_test_imp = imputer.transform(X_test_feat)
    
    # Scaling
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    
    # ==================== STEP 4: INITIALIZE ALGORITHMS ====================
    print("\n4. Initializing algorithms...")
    algorithms = create_comprehensive_algorithms()
    print(f"   Available algorithms: {len(algorithms)}")
    
    # Feature selection methods
    if quick_mode:
        feature_methods = ['rf_importance', 'lasso', 'elastic_net', 'mutual_info', 'f_classif', 'combined_filter']
        if XGB_AVAILABLE:
            feature_methods.append('xgb_importance')
    else:
        feature_methods = ['rf_importance', 'lasso', 'elastic_net', 'mutual_info', 'f_classif', 'combined_filter']
        if XGB_AVAILABLE:
            feature_methods.append('xgb_importance')
    
    # Imbalance methods
    if quick_mode:
        imbalance_methods = ['none', 'balanced', 'cost_5x']
        if IMBLEARN_AVAILABLE:
            imbalance_methods.extend(['smote', 'smote_enn'])
    else:
        imbalance_methods = ['none', 'balanced', 'cost_2x', 'cost_5x', 'cost_10x']
        if IMBLEARN_AVAILABLE:
            imbalance_methods.extend(['smote', 'smote_enn', 'borderline_smote', 'adasyn'])
    
    total_combinations = len(algorithms) * len(feature_methods) * len(imbalance_methods)
    print(f"\n5. Testing {total_combinations:,} combinations on VALIDATION set...")
    print(f"   {len(algorithms)} algorithms × {len(feature_methods)} features × {len(imbalance_methods)} imbalance")
    
    # ==================== STEP 5: EVALUATE ON VALIDATION SET ====================
    all_results = []
    combination_num = 0
    start_time = time.time()
    
    for algo_name, algorithm in algorithms.items():
        for feat_method in feature_methods:
            for imb_method in imbalance_methods:
                combination_num += 1
                
                if combination_num % 50 == 0:
                    elapsed = time.time() - start_time
                    eta = (elapsed / combination_num) * (total_combinations - combination_num)
                    print(f"   Progress: {combination_num}/{total_combinations} ({combination_num/total_combinations*100:.1f}%) - ETA: {eta/60:.1f} min")
                
                try:
                    # Feature selection - fit on train
                    selector = EnhancedFeatureSelector(method=feat_method, n_features=20)
                    selector.fit(X_train_scaled, y_train, sw_train)
                    X_train_selected = selector.transform(X_train_scaled)
                    X_val_selected = selector.transform(X_val_scaled)
                    
                    # Imbalance handling
                    X_train_final, y_train_final, sw_train_final = handle_enhanced_imbalance(
                        X_train_selected, y_train, sw_train, method=imb_method
                    )
                    
                    # Train model
                    clf = clone(algorithm)
                    
                    if algo_name in ['GaussianNB', 'LDA', 'KNN']:
                        clf.fit(X_train_final, y_train_final)
                    else:
                        try:
                            clf.fit(X_train_final, y_train_final, sample_weight=sw_train_final)
                        except:
                            clf.fit(X_train_final, y_train_final)
                    
                    # Calibrate if requested
                    if use_calibration:
                        try:
                            calibrated_clf = CalibratedClassifierCV(clf, method='isotonic', cv='prefit')
                            calibrated_clf.fit(X_val_selected, y_val)
                            clf = calibrated_clf
                        except:
                            pass
                    
                    # Predict on VALIDATION set
                    if hasattr(clf, 'predict_proba'):
                        y_val_proba = clf.predict_proba(X_val_selected)
                        if len(y_val_proba.shape) > 1 and y_val_proba.shape[1] > 1:
                            y_val_proba = y_val_proba[:, 1]
                    else:
                        try:
                            y_val_proba = clf.decision_function(X_val_selected)
                            y_val_proba = expit(y_val_proba)
                        except:
                            y_val_proba = clf.predict(X_val_selected)
                    
                    # Find optimal threshold on validation set
                    fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
                    idx = np.where(tpr >= min_sensitivity)[0]
                    if len(idx) > 0:
                        optimal_threshold = thresholds[idx[0]]
                    else:
                        optimal_threshold = 0.5
                    
                    # Calculate VALIDATION metrics
                    metrics = calculate_comprehensive_metrics(y_val, y_val_proba, sw_val,
                                                            threshold=optimal_threshold)
                    
                    result = {
                        'algorithm': algo_name, 
                        'features': feat_method, 
                        'imbalance': imb_method,
                        'clf': clf,  # Save for test evaluation
                        'selector': selector,  # Save for test evaluation
                        'threshold': optimal_threshold,
                        **metrics
                    }
                    all_results.append(result)
                    
                except Exception as e:
                    result = {
                        'algorithm': algo_name, 
                        'features': feat_method, 
                        'imbalance': imb_method, 
                        'auc': np.nan, 
                        'error': str(e)[:100]
                    }
                    all_results.append(result)
    
    results_df = pd.DataFrame(all_results)
    successful = results_df[~results_df['auc'].isna()].copy()
    
    print("\n" + "="*80)
    print("VALIDATION EVALUATION COMPLETE")
    print("="*80)
    print(f"Successful: {len(successful)}/{total_combinations}")
    print(f"Time: {(time.time()-start_time)/60:.1f} minutes")
    
    # ==================== STEP 6: SELECT BEST MODEL ON VALIDATION ====================
    if len(successful) > 0:
        successful['composite_score'] = (
            successful['auc'] * 0.25 + successful['sensitivity'] * 0.25 +
            successful['specificity'] * 0.15 + successful['ppv'] * 0.15 +
            successful['npv'] * 0.10 + (1 - successful.get('brier', 0.5)) * 0.05
        )
        
        print("\nTOP 10 MODELS ON VALIDATION SET:")
        print("-"*80)
        top_models = successful.nlargest(10, 'composite_score')
        for idx, row in top_models.iterrows():
            print(f"{row['algorithm']:20} + {row['features']:15} + {row['imbalance']:12} | "
                  f"AUC: {row['auc']:.3f}, Sens: {row['sensitivity']:.1%}, Spec: {row['specificity']:.1%}")
        
        # Select best high-sensitivity model
        high_sens = successful[successful['sensitivity'] >= min_sensitivity]
        if len(high_sens) > 0:
            print(f"\n{len(high_sens)} models ≥{min_sensitivity:.0%} sensitivity on validation")
            
            # Best model by specificity among high-sensitivity models
            best_model = high_sens.nlargest(1, 'specificity').iloc[0]
            
            print(f"\nSELECTED BEST MODEL (Validation Set):")
            print(f"  Algorithm: {best_model['algorithm']}")
            print(f"  Features: {best_model['features']}")
            print(f"  Imbalance: {best_model['imbalance']}")
            print(f"  Validation AUC: {best_model['auc']:.3f}")
            print(f"  Validation Sensitivity: {best_model['sensitivity']:.1%}")
            print(f"  Validation Specificity: {best_model['specificity']:.1%}")
            
            # ==================== STEP 7: EVALUATE ONCE ON TEST SET ====================
            print("\n" + "="*80)
            print("FINAL TEST SET EVALUATION (SINGLE MODEL)")
            print("="*80)
            
            # Get saved model and selector
            final_clf = best_model['clf']
            final_selector = best_model['selector']
            final_threshold = best_model['threshold']
            
            # Transform test set
            X_test_selected = final_selector.transform(X_test_scaled)
            
            # Predict on test set
            if hasattr(final_clf, 'predict_proba'):
                y_test_proba = final_clf.predict_proba(X_test_selected)
                if len(y_test_proba.shape) > 1 and y_test_proba.shape[1] > 1:
                    y_test_proba = y_test_proba[:, 1]
            else:
                try:
                    y_test_proba = final_clf.decision_function(X_test_selected)
                    y_test_proba = expit(y_test_proba)
                except:
                    y_test_proba = final_clf.predict(X_test_selected)
            
            # Calculate TEST metrics
            test_metrics = calculate_comprehensive_metrics(y_test, y_test_proba, sw_test,
                                                          threshold=final_threshold)
            
            print("\nFINAL TEST SET PERFORMANCE:")
            print(f"  AUC-ROC:        {test_metrics['auc']:.3f}")
            print(f"  Sensitivity:    {test_metrics['sensitivity']:.1%}")
            print(f"  Specificity:    {test_metrics['specificity']:.1%}")
            print(f"  PPV:            {test_metrics['ppv']:.1%}")
            print(f"  NPV:            {test_metrics['npv']:.1%}")
            print(f"  NNS:            {test_metrics['nns']:.1f}")
            print(f"  Net Benefit:    {test_metrics['net_benefit']:.3f}")
            print(f"  F1 Score:       {test_metrics['f1']:.3f}")
            if not pd.isna(test_metrics.get('mcc')):
                print(f"  MCC:            {test_metrics['mcc']:.3f}")
            if not pd.isna(test_metrics.get('brier')):
                print(f"  Brier Score:    {test_metrics['brier']:.3f}")
            
            # Create final results
            final_results = {
                'model_config': {
                    'algorithm': best_model['algorithm'],
                    'features': best_model['features'],
                    'imbalance': best_model['imbalance'],
                    'threshold': final_threshold
                },
                'validation_metrics': {
                    'auc': best_model['auc'],
                    'sensitivity': best_model['sensitivity'],
                    'specificity': best_model['specificity'],
                    'ppv': best_model['ppv'],
                    'npv': best_model['npv']
                },
                'test_metrics': test_metrics,
                'validation_leaderboard': successful[['algorithm', 'features', 'imbalance', 
                                                      'auc', 'sensitivity', 'specificity', 
                                                      'ppv', 'npv', 'composite_score']].copy()
            }
            
            return final_results
        else:
            print(f"\nNo models met {min_sensitivity:.0%} sensitivity threshold")
            return {'validation_leaderboard': successful}
    else:
        print("\nNo successful models")
        return None

# ===========================================================================
# MAIN EXECUTION
# ===========================================================================

if __name__ == "__main__":
    
    print("="*80)
    print("PEDIATRIC ASTHMA PIPELINE - NO DATA LEAKAGE")
    print("="*80)
    
    target_column = 'MCQ010'
    weight_column = 'WTMEC2YR'
    
    # Get features (excluding target and weight)
    feature_columns = [col for col in split_df.columns 
                       if col not in [target_column, weight_column]]
    
    X = split_df[feature_columns]
    y = split_df[target_column].copy()
    sample_weight = split_df[weight_column] if weight_column in split_df.columns else None
    
    # Convert target to binary
    if set(y.dropna().unique()) == {1, 2} or set(y.dropna().unique()) == {1.0, 2.0}:
        y = (y == 1).astype(float)
        y[split_df[target_column].isna()] = np.nan
    
    # Remove rows with missing target or invalid weights
    valid_mask = y.notna()
    if sample_weight is not None:
        valid_mask &= sample_weight.notna() & (sample_weight > 0)
    
    X_clean = X[valid_mask].reset_index(drop=True)
    y_clean = y[valid_mask].reset_index(drop=True)
    sample_weight_clean = sample_weight[valid_mask].reset_index(drop=True) if sample_weight is not None else None
    
    # Filter to pediatric age range
    if 'RIDAGEYR' in X_clean.columns:
        pediatric_mask = (X_clean['RIDAGEYR'] >= 6) & (X_clean['RIDAGEYR'] < 18)
        if pediatric_mask.sum() > 100:
            X_clean = X_clean[pediatric_mask].reset_index(drop=True)
            y_clean = y_clean[pediatric_mask].reset_index(drop=True)
            if sample_weight_clean is not None:
                sample_weight_clean = sample_weight_clean[pediatric_mask].reset_index(drop=True)
    
    print(f"Dataset: {len(X_clean):,} samples, {X_clean.shape[1]} features")
    print(f"Asthma prevalence: {np.average(y_clean, weights=sample_weight_clean):.1%}")
    
    # Run evaluation
    results = evaluate_comprehensive_pipeline_no_leakage(
        X_clean, y_clean, sample_weight_clean, 
        min_sensitivity=0.80,
        use_calibration=True,
        quick_mode=False  # Set False for full mode
    )
    
    # Save results
    if results is not None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Save validation leaderboard
        if 'validation_leaderboard' in results:
            leaderboard = results['validation_leaderboard']
            leaderboard_file = f'validation_leaderboard_{timestamp}.csv'
            leaderboard.to_csv(leaderboard_file, index=False)
            print(f"\nValidation leaderboard saved: {leaderboard_file}")
        
        # Save final test results
        if 'test_metrics' in results:
            import json
            final_file = f'final_test_results_{timestamp}.json'
            # Convert to serializable format
            save_results = {
                'model_config': results['model_config'],
                'validation_metrics': results['validation_metrics'],
                'test_metrics': {k: float(v) if not pd.isna(v) else None 
                               for k, v in results['test_metrics'].items()}
            }
            with open(final_file, 'w') as f:
                json.dump(save_results, f, indent=2)
            print(f"Final test results saved: {final_file}")
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETE - NO DATA LEAKAGE")
    print("="*80)

PEDIATRIC ASTHMA PIPELINE - NO DATA LEAKAGE
Dataset: 6,567 samples, 96 features
Asthma prevalence: 18.8%
COMPREHENSIVE PEDIATRIC ASTHMA PIPELINE - NO DATA LEAKAGE
Started at: 2026-04-26 10:32:44
Minimum sensitivity threshold: 80.0%
Mode: Full
--------------------------------------------------------------------------------

1. Removing leaky, age-restricted, and identifier columns...
   Remaining columns: 96

2. Splitting data BEFORE any feature engineering (60% train / 20% val / 20% test)...
   Training:   3,939 samples
   Validation: 1,314 samples
   Test:       1,314 samples
   Asthma prevalence: 18.9%

3. Fitting preprocessing transformers on TRAINING data only...
   Creating exhaustive clinical features...
   Total features after engineering: 210

4. Initializing algorithms...
   Available algorithms: 18

5. Testing 1,134 combinations on VALIDATION set...
   18 algorithms × 7 features × 9 imbalance
   Progress: 50/1134 (4.4%) - ETA: 247.2 min
   Progress: 100/1134 (8.8%) - ETA: 252

In [5]:
print(f"split_df shape: {split_df.shape}")
print(f"split_df dtypes summary: {split_df.dtypes.value_counts().to_dict()}")
print(f"MCQ010 in columns: {'MCQ010' in split_df.columns}")
print(f"WTMEC2YR in columns: {'WTMEC2YR' in split_df.columns}")

split_df shape: (6784, 98)
split_df dtypes summary: {dtype('float64'): 98}
MCQ010 in columns: True
WTMEC2YR in columns: True


In [6]:
import optuna
print(f"optuna version: {optuna.__version__}")

optuna version: 4.8.0


In [9]:
"""
STANDALONE HYPERPARAMETER TUNING FOR PEDIATRIC ASTHMA SCREENING
- Uses split_df from memory (does NOT modify it)
- Quiet mode: only shows progress summaries
- Tunes 3 models: CatBoost, MLP, BalancedRF
"""

import numpy as np
import pandas as pd
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import make_scorer, recall_score, roc_auc_score, precision_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neural_network import MLPClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
from catboost import CatBoostClassifier
import joblib
from datetime import datetime
import warnings
import os
import time
warnings.filterwarnings('ignore')

# Suppress Optuna's per-trial logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ===========================================================================
# CONFIGURATION
# ===========================================================================

# Data source: split_df from memory (WILL NOT BE MODIFIED)
TARGET_COLUMN = 'MCQ010'  # Asthma diagnosis
WEIGHT_COLUMN = 'WTMEC2YR'  # Survey weights

# Leaky variables (must be removed)
LEAKY_PROXIES = ["MCQ025", "MCQ035", "MCQ040", "MCQ050", "MCQ051"]
AGE_RESTRICTED_VARS = [
    'ECQ020', 'ECQ080', 'ECQ090', 'WHQ030E', 'MCQ080E', 
    'ECQ150', 'ECD010', 'ECD070A', 'ECD070B',
    'FSD670ZC', 'FSQ690', 'FSD680', 'FSD675'
]
IDENTIFIERS = ['SEQN']

# OPTUNA SETTINGS
N_TRIALS = 100  # SMOKE TEST — change back to 100 for real run
N_JOBS = -1  # Use all CPU cores
RANDOM_STATE = 42
CV_FOLDS = 5

# CONSTRAINTS
MIN_SENSITIVITY = 0.80
MIN_SPECIFICITY = 0.70

# OUTPUT
OUTPUT_DIR = f"tuning_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*80)
print("STANDALONE HYPERPARAMETER TUNING FOR PEDIATRIC ASTHMA SCREENING")
print("="*80)
print(f"\nConfiguration:")
print(f"  • Data source: split_df (COPY - original not modified)")
print(f"  • Trials per model: {N_TRIALS}")
print(f"  • Cross-validation: {CV_FOLDS}-fold")
print(f"  • Min sensitivity: {MIN_SENSITIVITY}")
print(f"  • Min specificity: {MIN_SPECIFICITY}")
print(f"  • Output directory: {OUTPUT_DIR}/")
print(f"  • Random state: {RANDOM_STATE}")

# ===========================================================================
# NHANES DATA CLEANING
# ===========================================================================

class NHANESCleaner(BaseEstimator, TransformerMixin):
    """Clean NHANES sentinel codes"""
    
    def __init__(self):
        self.continuous_cols_ = None
        self.categorical_cols_ = None
        self.binary_cols_ = None
        
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        continuous_whitelist = {
            'BMXBMI', 'BMXHT', 'BMXWT', 'BMXWAIST', 'BMXLEG', 'BMXARML', 'BMXARMC',
            'SPXNFEV1', 'SPXNFVC', 'SPXNFEV3', 'SPXNFEV6', 'SPXNPEF', 'SPXBFEV1', 'SPXBFVC',
            'LBXCOT', 'LBXWBCSI', 'LBXEOPCT', 'INDFMPIR', 'RIDAGEYR'
        }
        
        self.continuous_cols_ = []
        self.categorical_cols_ = []
        self.binary_cols_ = []
        
        for col in X_df.columns:
            if col in continuous_whitelist:
                self.continuous_cols_.append(col)
            else:
                s = X_df[col].dropna()
                if len(s) > 0 and pd.api.types.is_numeric_dtype(X_df[col]):
                    unique_vals = set(s.unique())
                    if unique_vals == {1, 2} or unique_vals == {1.0, 2.0}:
                        self.binary_cols_.append(col)
                    elif np.all(np.abs(s - np.round(s)) < 1e-6) and len(s.unique()) <= 20:
                        self.categorical_cols_.append(col)
                    else:
                        self.continuous_cols_.append(col)
        
        return self
    
    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        big_sentinels = {99, 999, 777, 7777, 9999}
        small_sentinels = {7, 9, 77}
        
        for col in self.continuous_cols_:
            if col in X_df.columns:
                X_df[col] = X_df[col].replace(big_sentinels, np.nan)
        
        for col in self.categorical_cols_:
            if col in X_df.columns:
                X_df[col] = X_df[col].replace(big_sentinels | small_sentinels, np.nan)
        
        for col in self.binary_cols_:
            if col in X_df.columns:
                X_df[col] = (X_df[col] == 1).astype(float)
                X_df.loc[X[col].isna(), col] = np.nan
        
        return X_df

# ===========================================================================
# FEATURE ENGINEERING
# ===========================================================================

class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    """Create clinical features"""
    
    def __init__(self):
        self.bmi_mean_ = None
        self.bmi_std_ = None
        self.feature_names_ = None
        
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        if 'BMXBMI' in X_df.columns:
            self.bmi_mean_ = X_df['BMXBMI'].mean()
            self.bmi_std_ = X_df['BMXBMI'].std()
            if self.bmi_std_ == 0 or pd.isna(self.bmi_std_):
                self.bmi_std_ = 1.0
        
        return self
    
    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Spirometry features
        if 'SPXNFEV1' in X_df.columns and 'SPXNFVC' in X_df.columns:
            mask = (X_df['SPXNFEV1'].notna() & X_df['SPXNFVC'].notna() & (X_df['SPXNFVC'] > 0))
            X_df['fev1_fvc_ratio'] = np.nan
            X_df.loc[mask, 'fev1_fvc_ratio'] = X_df.loc[mask, 'SPXNFEV1'] / X_df.loc[mask, 'SPXNFVC']
            X_df['obstruction_indicator'] = (X_df['fev1_fvc_ratio'] < 0.8).astype(float)
            X_df['fev1_log'] = np.log1p(X_df['SPXNFEV1'].fillna(0))
        
        # Cotinine features
        if 'LBXCOT' in X_df.columns:
            X_df['cotinine_log'] = np.log1p(X_df['LBXCOT'].fillna(0))
            X_df['smoke_exposure_heavy'] = (X_df['LBXCOT'] > 10.0).astype(float)
            X_df['likely_active_smoking'] = (X_df['LBXCOT'] > 50).astype(float)
        
        # BMI features
        if 'BMXBMI' in X_df.columns:
            X_df['bmi_log'] = np.log1p(X_df['BMXBMI'].fillna(0))
            X_df['obese'] = (X_df['BMXBMI'] >= 30).astype(float)
            if self.bmi_mean_ is not None and self.bmi_std_ is not None:
                X_df['bmi_zscore'] = (X_df['BMXBMI'] - self.bmi_mean_) / self.bmi_std_
        
        # Age features
        if 'RIDAGEYR' in X_df.columns:
            X_df['age_squared'] = X_df['RIDAGEYR'] ** 2
            X_df['age_log'] = np.log1p(X_df['RIDAGEYR'])
        
        # Family history interactions
        if 'MCQ300B' in X_df.columns and 'fev1_fvc_ratio' in X_df.columns:
            X_df['family_spirometry_interaction'] = X_df['MCQ300B'].fillna(0) * X_df['fev1_fvc_ratio'].fillna(0)
        
        # Missing indicators
        for var in ['SPXNFEV1', 'SPXNFVC', 'LBXCOT', 'BMXBMI']:
            if var in X_df.columns:
                X_df[f'{var}_missing'] = X_df[var].isna().astype(int)
        
        self.feature_names_ = X_df.columns.tolist()
        return X_df
    
    def get_feature_names_out(self, input_features=None):
        return self.feature_names_ if self.feature_names_ is not None else []

# ===========================================================================
# LOAD AND PREPARE DATA
# ===========================================================================

print("\n" + "-"*80)
print("LOADING AND PREPARING DATA")
print("-"*80)

# Use split_df from memory - CREATE A COPY (original is NOT modified)
try:
    print("Creating copy of split_df (original remains unchanged)...")
    df = split_df.copy()
    print(f"✓ Loaded: {len(df):,} rows, {df.shape[1]} columns")
except NameError:
    print("❌ ERROR: split_df not found in memory!")
    print("\nPlease ensure you have the comprehensive pipeline's split_df available.")
    raise

# Extract target and features
y = df[TARGET_COLUMN].copy()
sample_weight = df[WEIGHT_COLUMN] if WEIGHT_COLUMN in df.columns else None
feature_columns = [col for col in df.columns if col not in [TARGET_COLUMN, WEIGHT_COLUMN]]
X = df[feature_columns].copy()

# Convert target to binary
if set(y.dropna().unique()) == {1, 2} or set(y.dropna().unique()) == {1.0, 2.0}:
    y = (y == 1).astype(float)
    y[df[TARGET_COLUMN].isna()] = np.nan

# Remove invalid rows
valid_mask = y.notna()
if sample_weight is not None:
    valid_mask &= sample_weight.notna() & (sample_weight > 0)

X_clean = X[valid_mask].reset_index(drop=True)
y_clean = y[valid_mask].reset_index(drop=True)
sample_weight_clean = sample_weight[valid_mask].reset_index(drop=True) if sample_weight is not None else None

# Filter to pediatric age (6-17)
if 'RIDAGEYR' in X_clean.columns:
    pediatric_mask = (X_clean['RIDAGEYR'] >= 6) & (X_clean['RIDAGEYR'] < 18)
    X_clean = X_clean[pediatric_mask].reset_index(drop=True)
    y_clean = y_clean[pediatric_mask].reset_index(drop=True)
    if sample_weight_clean is not None:
        sample_weight_clean = sample_weight_clean[pediatric_mask].reset_index(drop=True)

print(f"Valid pediatric samples: {len(X_clean):,}")
print(f"Asthma prevalence: {y_clean.mean():.1%} (n={int(y_clean.sum())})")

# Remove leaky/restricted columns
cols_to_drop = []
for col in X_clean.columns:
    if col in LEAKY_PROXIES + AGE_RESTRICTED_VARS + IDENTIFIERS:
        cols_to_drop.append(col)

if cols_to_drop:
    X_clean = X_clean.drop(columns=cols_to_drop)

print(f"Features before engineering: {X_clean.shape[1]}")

# Split data (60% train, 20% val, 20% test)
if sample_weight_clean is not None:
    X_temp, X_test, y_temp, y_test, sw_temp, sw_test = train_test_split(
        X_clean, y_clean, sample_weight_clean, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clean
    )
    X_train, X_val, y_train, y_val, sw_train, sw_val = train_test_split(
        X_temp, y_temp, sw_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
    )
else:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clean
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
    )
    sw_train = sw_val = sw_test = None

print(f"Training:   {len(X_train):,} samples ({y_train.mean():.1%} asthma)")
print(f"Validation: {len(X_val):,} samples ({y_val.mean():.1%} asthma)")
print(f"Test:       {len(X_test):,} samples ({y_test.mean():.1%} asthma)")

# Preprocessing pipeline
print("\nApplying preprocessing (cleaning → feature engineering → imputation → scaling)...")
cleaner = NHANESCleaner()
feat_eng = ClinicalFeatureEngineer()
imputer = SimpleImputer(strategy='median')
scaler = RobustScaler()

X_train_clean = cleaner.fit_transform(X_train)
X_train_feat = feat_eng.fit_transform(X_train_clean)
X_train_imp = imputer.fit_transform(X_train_feat)
X_train_final = scaler.fit_transform(X_train_imp)

X_val_clean = cleaner.transform(X_val)
X_val_feat = feat_eng.transform(X_val_clean)
X_val_imp = imputer.transform(X_val_feat)
X_val_final = scaler.transform(X_val_imp)

X_test_clean = cleaner.transform(X_test)
X_test_feat = feat_eng.transform(X_test_clean)
X_test_imp = imputer.transform(X_test_feat)
X_test_final = scaler.transform(X_test_imp)

print(f"✓ Preprocessing complete! Final features: {X_train_final.shape[1]}")

# Save preprocessed data
joblib.dump({
    'cleaner': cleaner,
    'feat_eng': feat_eng,
    'imputer': imputer,
    'scaler': scaler,
    'X_train': X_train_final,
    'y_train': y_train,
    'X_val': X_val_final,
    'y_val': y_val,
    'X_test': X_test_final,
    'y_test': y_test,
    'sw_train': sw_train,
    'sw_val': sw_val,
    'sw_test': sw_test,
    'feature_names': feat_eng.feature_names_
}, f'{OUTPUT_DIR}/preprocessed_data.pkl')

# ===========================================================================
# MODEL 1: CATBOOST TUNING
# ===========================================================================

print("\n" + "="*80)
print("MODEL 1: CATBOOST HYPERPARAMETER TUNING")
print("="*80)

SCORING = {
    'sensitivity': make_scorer(recall_score),
    'specificity': make_scorer(recall_score, pos_label=0),
    'auc': 'roc_auc'
}

def objective_catboost(trial):
    """Optuna objective for CatBoost"""
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'random_state': RANDOM_STATE,
        'verbose': False,
        'thread_count': 1
    }
    
    pipeline = ImbPipeline([
        ('feature_selection', SelectKBest(f_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('classifier', CatBoostClassifier(**params))
    ])
    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    try:
        cv_results = cross_validate(
            pipeline, X_train_final, y_train,
            cv=cv, scoring=SCORING, n_jobs=1, error_score='raise'
        )
        
        mean_sensitivity = cv_results['test_sensitivity'].mean()
        mean_specificity = cv_results['test_specificity'].mean()
        mean_auc = cv_results['test_auc'].mean()
        
        trial.set_user_attr('specificity', mean_specificity)
        trial.set_user_attr('auc', mean_auc)
        trial.set_user_attr('sensitivity_std', cv_results['test_sensitivity'].std())
        
        if mean_specificity < MIN_SPECIFICITY:
            penalty = (MIN_SPECIFICITY - mean_specificity) * 2
            return mean_sensitivity - penalty
        
        return mean_sensitivity
    except:
        return 0.0

study_catboost = optuna.create_study(
    direction='maximize',
    study_name='catboost_optimization',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print(f"Optimizing {N_TRIALS} trials (this may take 1-2 hours)...")
start_time = time.time()

study_catboost.optimize(objective_catboost, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=False)

elapsed = time.time() - start_time
print(f"✅ Complete ({elapsed/60:.1f} min)")

successful_trials_cb = [t for t in study_catboost.trials if t.value is not None and t.value > 0]
print(f"Successful: {len(successful_trials_cb)}/{len(study_catboost.trials)} trials")

if len(successful_trials_cb) > 0:
    print(f"Best - Sens: {study_catboost.best_value:.3f}, Spec: {study_catboost.best_trial.user_attrs.get('specificity', 0):.3f}, AUC: {study_catboost.best_trial.user_attrs.get('auc', 0):.3f}")
    
    joblib.dump(study_catboost, f'{OUTPUT_DIR}/catboost_study.pkl')
    study_catboost.trials_dataframe().to_csv(f'{OUTPUT_DIR}/catboost_trials.csv', index=False)
    
    best_params_cb = study_catboost.best_params.copy()
    best_params_cb.update({'random_state': RANDOM_STATE, 'verbose': False, 'thread_count': -1})
    
    final_pipeline_cb = ImbPipeline([
        ('feature_selection', SelectKBest(f_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('classifier', CatBoostClassifier(**best_params_cb))
    ])
    
    final_pipeline_cb.fit(X_train_final, y_train)
    joblib.dump(final_pipeline_cb, f'{OUTPUT_DIR}/catboost_best_model.pkl')
    CATBOOST_FAILED = False
else:
    print("⚠️  All trials failed")
    CATBOOST_FAILED = True

# ===========================================================================
# MODEL 2: MLP TUNING
# ===========================================================================

print("\n" + "="*80)
print("MODEL 2: MLP HYPERPARAMETER TUNING")
print("="*80)

class_counts = np.bincount(y_train.astype(int))
weight_0 = 1.0
weight_1 = (len(y_train) / (2 * class_counts[1])) * 5
CLASS_WEIGHTS = {0: weight_0, 1: weight_1}

def suggest_hidden_layers(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    return tuple([trial.suggest_int(f'n_units_l{i}', 32, 256) for i in range(n_layers)])

def objective_mlp(trial):
    """Optuna objective for MLP"""
    hidden_layer_sizes = suggest_hidden_layers(trial)
    
    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
        'max_iter': trial.suggest_int('max_iter', 300, 1000, step=100),
        'early_stopping': True,
        'validation_fraction': 0.15,
        'n_iter_no_change': 15,
        'random_state': RANDOM_STATE,
        'verbose': False
    }
    
    pipeline = ImbPipeline([
        ('feature_selection', SelectKBest(mutual_info_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('classifier', MLPClassifier(**params))
    ])
    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    try:
        cv_results = cross_validate(
            pipeline, X_train_final, y_train,
            cv=cv, scoring=SCORING, n_jobs=1,
            error_score='raise'
        )
        
        mean_sensitivity = cv_results['test_sensitivity'].mean()
        mean_specificity = cv_results['test_specificity'].mean()
        mean_auc = cv_results['test_auc'].mean()
        
        trial.set_user_attr('specificity', mean_specificity)
        trial.set_user_attr('auc', mean_auc)
        trial.set_user_attr('sensitivity_std', cv_results['test_sensitivity'].std())
        
        if mean_specificity < MIN_SPECIFICITY:
            penalty = (MIN_SPECIFICITY - mean_specificity) * 2
            return mean_sensitivity - penalty
        
        return mean_sensitivity
    except Exception as e:
        print(f"MLP trial failed: {e}")
        return 0.0

study_mlp = optuna.create_study(
    direction='maximize',
    study_name='mlp_optimization',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print(f"Optimizing {N_TRIALS} trials (this may take 40-80 minutes)...")
start_time = time.time()

study_mlp.optimize(objective_mlp, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=False)

elapsed = time.time() - start_time
print(f"✅ Complete ({elapsed/60:.1f} min)")

successful_trials_mlp = [t for t in study_mlp.trials if t.value is not None and t.value > 0]
print(f"Successful: {len(successful_trials_mlp)}/{len(study_mlp.trials)} trials")

if len(successful_trials_mlp) == 0:
    print("⚠️  All trials failed (common with MLP on small/imbalanced data)")
    MLP_FAILED = True
else:
    MLP_FAILED = False
    print(f"Best - Sens: {study_mlp.best_value:.3f}, Spec: {study_mlp.best_trial.user_attrs.get('specificity', 0):.3f}, AUC: {study_mlp.best_trial.user_attrs.get('auc', 0):.3f}")
    
    joblib.dump(study_mlp, f'{OUTPUT_DIR}/mlp_study.pkl')
    study_mlp.trials_dataframe().to_csv(f'{OUTPUT_DIR}/mlp_trials.csv', index=False)
    
    best_params_mlp = study_mlp.best_params.copy()
    n_layers = best_params_mlp.pop('n_layers')
    hidden_layer_sizes = tuple([best_params_mlp.pop(f'n_units_l{i}') for i in range(n_layers)])
    
    final_params_mlp = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': best_params_mlp['activation'],
        'alpha': best_params_mlp['alpha'],
        'learning_rate': best_params_mlp['learning_rate'],
        'learning_rate_init': best_params_mlp['learning_rate_init'],
        'max_iter': best_params_mlp['max_iter'],
        'early_stopping': True,
        'validation_fraction': 0.15,
        'n_iter_no_change': 15,
        'random_state': RANDOM_STATE,
        'verbose': False
    }
    
    final_pipeline_mlp = ImbPipeline([
        ('feature_selection', SelectKBest(mutual_info_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('classifier', MLPClassifier(**final_params_mlp))
    ])
    
    try:
        final_pipeline_mlp.fit(X_train_final, y_train)
        joblib.dump(final_pipeline_mlp, f'{OUTPUT_DIR}/mlp_best_model.pkl')
    except Exception as e:
        print(f"Final MLP fit failed: {e}")
        MLP_FAILED = True

# ===========================================================================
# MODEL 3: BALANCED RANDOM FOREST TUNING
# ===========================================================================

print("\n" + "="*80)
print("MODEL 3: BALANCED RANDOM FOREST HYPERPARAMETER TUNING")
print("="*80)

def objective_balancedrf(trial):
    """Optuna objective for Balanced RF"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'sampling_strategy': trial.suggest_categorical('sampling_strategy', ['not minority', 'not majority', 'all']),
        'random_state': RANDOM_STATE,
        'n_jobs': 1,
        'verbose': 0
    }
    
    if params['bootstrap']:
        params['max_samples'] = trial.suggest_float('max_samples', 0.5, 1.0)
    
    pipeline = ImbPipeline([
        ('feature_selection', SelectKBest(f_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('classifier', BalancedRandomForestClassifier(**params))
    ])
    
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    try:
        cv_results = cross_validate(
            pipeline, X_train_final, y_train,
            cv=cv, scoring=SCORING, n_jobs=1, error_score='raise'
        )
        
        mean_sensitivity = cv_results['test_sensitivity'].mean()
        mean_specificity = cv_results['test_specificity'].mean()
        mean_auc = cv_results['test_auc'].mean()
        
        trial.set_user_attr('specificity', mean_specificity)
        trial.set_user_attr('auc', mean_auc)
        trial.set_user_attr('sensitivity_std', cv_results['test_sensitivity'].std())
        
        if mean_specificity < MIN_SPECIFICITY:
            penalty = (MIN_SPECIFICITY - mean_specificity) * 2
            return mean_sensitivity - penalty
        
        return mean_sensitivity
    except:
        return 0.0

study_rf = optuna.create_study(
    direction='maximize',
    study_name='balancedrf_optimization',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

print(f"Optimizing {N_TRIALS} trials (this may take 1.5-2.5 hours)...")
start_time = time.time()

study_rf.optimize(objective_balancedrf, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=False)

elapsed = time.time() - start_time
print(f"✅ Complete ({elapsed/60:.1f} min)")

successful_trials_rf = [t for t in study_rf.trials if t.value is not None and t.value > 0]
print(f"Successful: {len(successful_trials_rf)}/{len(study_rf.trials)} trials")

if len(successful_trials_rf) > 0:
    print(f"Best - Sens: {study_rf.best_value:.3f}, Spec: {study_rf.best_trial.user_attrs.get('specificity', 0):.3f}, AUC: {study_rf.best_trial.user_attrs.get('auc', 0):.3f}")
    
    joblib.dump(study_rf, f'{OUTPUT_DIR}/balancedrf_study.pkl')
    study_rf.trials_dataframe().to_csv(f'{OUTPUT_DIR}/balancedrf_trials.csv', index=False)
    
    best_params_rf = study_rf.best_params.copy()
    best_params_rf.update({'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': 0})
    
    if not best_params_rf['bootstrap']:
        best_params_rf.pop('max_samples', None)
    
    final_pipeline_rf = ImbPipeline([
        ('feature_selection', SelectKBest(f_classif, k=20)),
        ('smote_enn', SMOTEENN(random_state=RANDOM_STATE)),
        ('classifier', BalancedRandomForestClassifier(**best_params_rf))
    ])
    
    final_pipeline_rf.fit(X_train_final, y_train)
    joblib.dump(final_pipeline_rf, f'{OUTPUT_DIR}/balancedrf_best_model.pkl')
    BALANCEDRF_FAILED = False
else:
    print("⚠️  All trials failed")
    BALANCEDRF_FAILED = True

# ===========================================================================
# GENERATE VISUALIZATIONS
# ===========================================================================

print("\n" + "-"*80)
print("Generating visualizations...")
print("-"*80)

sns.set_style("whitegrid")

studies_to_viz = []
if not CATBOOST_FAILED:
    studies_to_viz.append((study_catboost, 'CatBoost'))
if not MLP_FAILED:
    studies_to_viz.append((study_mlp, 'MLP'))
if not BALANCEDRF_FAILED:
    studies_to_viz.append((study_rf, 'BalancedRF'))

for study, name in studies_to_viz:
    try:
        fig1 = plot_optimization_history(study)
        fig1.update_layout(title=f"{name} Optimization History")
        fig1.write_html(f'{OUTPUT_DIR}/{name.lower()}_optimization_history.html')
        
        fig2 = plot_param_importances(study)
        fig2.update_layout(title=f"{name} Hyperparameter Importances")
        fig2.write_html(f'{OUTPUT_DIR}/{name.lower()}_param_importances.html')
        
        fig, ax = plt.subplots(figsize=(10, 8))
        valid_trials = [t for t in study.trials if 'specificity' in t.user_attrs and t.value is not None]
        
        if len(valid_trials) > 0:
            sensitivities = [t.value for t in valid_trials]
            specificities = [t.user_attrs['specificity'] for t in valid_trials]
            
            scatter = ax.scatter(sensitivities, specificities, c=range(len(valid_trials)), 
                                cmap='viridis', alpha=0.6, s=50)
            
            best_sens = study.best_value
            best_spec = study.best_trial.user_attrs.get('specificity', 0)
            ax.scatter([best_sens], [best_spec], c='red', s=300, marker='*', 
                      edgecolors='black', linewidth=2, label='Best Trial', zorder=5)
            
            ax.axhline(y=MIN_SPECIFICITY, color='blue', linestyle='--', linewidth=2, 
                      label=f'Min Specificity ({MIN_SPECIFICITY})')
            ax.axvline(x=MIN_SENSITIVITY, color='green', linestyle='--', linewidth=2, 
                      label=f'Min Sensitivity ({MIN_SENSITIVITY})')
            
            ax.set_xlabel('Sensitivity', fontsize=12)
            ax.set_ylabel('Specificity', fontsize=12)
            ax.set_title(f'{name}: Sensitivity vs Specificity Trade-off', fontsize=14)
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            plt.colorbar(scatter, ax=ax, label='Trial Number')
            plt.tight_layout()
            plt.savefig(f'{OUTPUT_DIR}/{name.lower()}_sens_spec_tradeoff.png', dpi=300)
            plt.close()
    except:
        pass

print("✓ Visualizations saved")

# ===========================================================================
# SUMMARY REPORT
# ===========================================================================

print("\n" + "-"*80)
print("Generating summary report...")
print("-"*80)

with open(f'{OUTPUT_DIR}/tuning_summary.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("HYPERPARAMETER TUNING SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Trials per model: {N_TRIALS}\n")
    f.write(f"Cross-validation: {CV_FOLDS}-fold\n")
    f.write(f"Training samples: {len(X_train):,}\n\n")
    
    if not CATBOOST_FAILED:
        f.write("\nCATBOOST\n" + "-"*80 + "\n")
        f.write("Configuration: f_classif + smote_enn\n")
        f.write(f"Best Trial: #{study_catboost.best_trial.number}\n")
        f.write(f"Sensitivity (CV): {study_catboost.best_value:.4f}\n")
        f.write(f"Specificity (CV): {study_catboost.best_trial.user_attrs.get('specificity', 0):.4f}\n")
        f.write(f"AUC (CV): {study_catboost.best_trial.user_attrs.get('auc', 0):.4f}\n")
        f.write("\nBest Hyperparameters:\n")
        for key, value in study_catboost.best_params.items():
            f.write(f"  {key}: {value}\n")
    
    if not MLP_FAILED:
        f.write("\n\nMLP\n" + "-"*80 + "\n")
        f.write("Configuration: mutual_info + cost_5x\n")
        f.write(f"Best Trial: #{study_mlp.best_trial.number}\n")
        f.write(f"Sensitivity (CV): {study_mlp.best_value:.4f}\n")
        f.write(f"Specificity (CV): {study_mlp.best_trial.user_attrs.get('specificity', 0):.4f}\n")
        f.write(f"AUC (CV): {study_mlp.best_trial.user_attrs.get('auc', 0):.4f}\n")
        f.write("\nBest Hyperparameters:\n")
        for key, value in study_mlp.best_params.items():
            f.write(f"  {key}: {value}\n")
    
    if not BALANCEDRF_FAILED:
        f.write("\n\nBALANCED RANDOM FOREST\n" + "-"*80 + "\n")
        f.write("Configuration: f_classif + smote_enn\n")
        f.write(f"Best Trial: #{study_rf.best_trial.number}\n")
        f.write(f"Sensitivity (CV): {study_rf.best_value:.4f}\n")
        f.write(f"Specificity (CV): {study_rf.best_trial.user_attrs.get('specificity', 0):.4f}\n")
        f.write(f"AUC (CV): {study_rf.best_trial.user_attrs.get('auc', 0):.4f}\n")
        f.write("\nBest Hyperparameters:\n")
        for key, value in study_rf.best_params.items():
            f.write(f"  {key}: {value}\n")

# ===========================================================================
# FINAL COMPARISON
# ===========================================================================

comparison_data = []

if not CATBOOST_FAILED:
    comparison_data.append({
        'Model': 'CatBoost',
        'Sensitivity_CV': study_catboost.best_value,
        'Specificity_CV': study_catboost.best_trial.user_attrs.get('specificity', np.nan),
        'AUC_CV': study_catboost.best_trial.user_attrs.get('auc', np.nan),
        'Configuration': 'f_classif + smote_enn',
        'Status': 'Success'
    })
else:
    comparison_data.append({
        'Model': 'CatBoost', 'Sensitivity_CV': np.nan, 'Specificity_CV': np.nan,
        'AUC_CV': np.nan, 'Configuration': 'f_classif + smote_enn', 'Status': 'Failed'
    })

if not MLP_FAILED:
    comparison_data.append({
        'Model': 'MLP',
        'Sensitivity_CV': study_mlp.best_value,
        'Specificity_CV': study_mlp.best_trial.user_attrs.get('specificity', np.nan),
        'AUC_CV': study_mlp.best_trial.user_attrs.get('auc', np.nan),
        'Configuration': 'mutual_info + cost_5x',
        'Status': 'Success'
    })
else:
    comparison_data.append({
        'Model': 'MLP', 'Sensitivity_CV': np.nan, 'Specificity_CV': np.nan,
        'AUC_CV': np.nan, 'Configuration': 'mutual_info + cost_5x', 'Status': 'Failed'
    })

if not BALANCEDRF_FAILED:
    comparison_data.append({
        'Model': 'BalancedRF',
        'Sensitivity_CV': study_rf.best_value,
        'Specificity_CV': study_rf.best_trial.user_attrs.get('specificity', np.nan),
        'AUC_CV': study_rf.best_trial.user_attrs.get('auc', np.nan),
        'Configuration': 'f_classif + smote_enn',
        'Status': 'Success'
    })
else:
    comparison_data.append({
        'Model': 'BalancedRF', 'Sensitivity_CV': np.nan, 'Specificity_CV': np.nan,
        'AUC_CV': np.nan, 'Configuration': 'f_classif + smote_enn', 'Status': 'Failed'
    })

results_comparison = pd.DataFrame(comparison_data)
results_comparison.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)

# ===========================================================================
# FINAL SUMMARY
# ===========================================================================

print("\n" + "="*80)
print("✅ HYPERPARAMETER TUNING COMPLETE")
print("="*80)

successful_models = results_comparison[results_comparison['Status'] == 'Success']['Model'].tolist()
failed_models = results_comparison[results_comparison['Status'] == 'Failed']['Model'].tolist()

print(f"\n✓ Successful models ({len(successful_models)}): {', '.join(successful_models)}")
if failed_models:
    print(f"✗ Failed models ({len(failed_models)}): {', '.join(failed_models)}")

print(f"\n📁 All results saved to: {OUTPUT_DIR}/")
print("\n📊 Next steps:")
print(f"   1. Review {OUTPUT_DIR}/tuning_summary.txt")
print(f"   2. Open {OUTPUT_DIR}/*_optimization_history.html in browser")
print(f"   3. Load best model: joblib.load('{OUTPUT_DIR}/catboost_best_model.pkl')")

print("\n" + "="*80)

STANDALONE HYPERPARAMETER TUNING FOR PEDIATRIC ASTHMA SCREENING

Configuration:
  • Data source: split_df (COPY - original not modified)
  • Trials per model: 100
  • Cross-validation: 5-fold
  • Min sensitivity: 0.8
  • Min specificity: 0.7
  • Output directory: tuning_results_20260426_180035/
  • Random state: 42

--------------------------------------------------------------------------------
LOADING AND PREPARING DATA
--------------------------------------------------------------------------------
Creating copy of split_df (original remains unchanged)...
✓ Loaded: 6,784 rows, 98 columns
Valid pediatric samples: 6,567
Asthma prevalence: 18.7% (n=1229)
Features before engineering: 96
Training:   3,939 samples (18.7% asthma)
Validation: 1,314 samples (18.7% asthma)
Test:       1,314 samples (18.7% asthma)

Applying preprocessing (cleaning → feature engineering → imputation → scaling)...
✓ Preprocessing complete! Final features: 112

MODEL 1: CATBOOST HYPERPARAMETER TUNING
Optimizing 1